# 🛸 Orbit Wars: Advanced Relational Transformer PPO Training Pipeline
This notebook implements the cloud-accelerated execution pipeline for training a state-of-the-art multi-agent reinforcement learning agent in the continuous-space **Orbit Wars** environment.

### 🛠️ Strategic Architecture Mapping
1. **State Abstraction**: 18-dimensional predictive token vectors tracking travel times and destination arrival garrisons. Preserves internal entity IDs to expose engine list-order collision biases (*Geometric Tunneling*).
2. **Policy Configuration**: An $N \times N$ Relational Source-Target Transformer that processes multi-front joint orchestration.
3. **Exploration & Stability**: Initialized via Supervised Behavioral Cloning (BC) from expert heuristics, fine-tuned online via Proximal Policy Optimization (PPO) using Potential-Based Reward Shaping (PBRS) to prevent reward hacking.


In [ ]:
# @title Step 1: Persistent Google Drive & Environment Initialization
import os
import sys
from google.colab import drive

print("Mounting Google Drive boundary...")
drive.mount('/content/drive')

# @markdown Define the path to your 'Orbit_wars' folder in Google Drive:
PROJECT_PATH = "/content/drive/MyDrive/Orbit_wars" # @param {type:"string"}

if not os.path.exists(PROJECT_PATH):
    print(f"❌ ERROR: Path not found in Drive: {PROJECT_PATH}")
    print("Please ensure you have uploaded the 'Orbit_wars' folder to your Drive root.")
else:
    %cd $PROJECT_PATH
    print(f"✅ Workspace synchronized at: {PROJECT_PATH}")

    # 1. Install core compilation dependencies
    !pip install -r requirements.txt --quiet
    !pip install kaggle-environments --upgrade --quiet

    # 2. Inject root path directory to enforce absolute system package lookups
    sys.path.append(os.path.abspath("."))

    import torch
    print(f"CUDA Accelerator Status: {'ACTIVE (GPU Enabled)' if torch.cuda.is_available() else 'INACTIVE (CPU Mode)'}")
    
    # Create backup folder for checkpoints if missing
    os.makedirs('/content/drive/MyDrive/OrbitWars_Checkpoints', exist_ok=True)

In [ ]:
# @title 📂 Step 1b: Restore Expert Trajectories from Drive
# Manual code uploads to Drive can delete your 'data/' folder.
# Run this to pull your accumulated transitions back into the project workspace.
import shutil
import os

SOURCE_NPZ = "/content/drive/MyDrive/OrbitWars_Checkpoints/bc_data.npz"
DEST_DIR = "data/bc_dataset"
DEST_FILE = os.path.join(DEST_DIR, "bc_data.npz")

if os.path.exists(SOURCE_NPZ):
    os.makedirs(DEST_DIR, exist_ok=True)
    # Use shutil.copy2 to preserve metadata
    shutil.copy2(SOURCE_NPZ, DEST_FILE)
    file_size = os.path.getsize(DEST_FILE) / (1024 * 1024)
    print(f"✅ Data Synchronized: {DEST_FILE} ({file_size:.2f} MB)")
    print("You can now resume harvesting or start training!")
else:
    print("ℹ️ No existing bc_data.npz found in backup. Ready for fresh collection.")

## 🏭 Phase 2: Supervised Expert Trajectory Harvesting
To bypass uninformative exploration traps (where random actions instantly vaporize fleets into the central Sun), we execute a data collection sprint. This rolls out the highly tuned rule-based `HeuristicBaseline` agent to gather 100,000 high-fidelity transitions.

The trajectory compiler cross-references heuristic vectors against our wrapper's 8-iteration numerical intercept solver to match precise discrete target token labels.


In [ ]:
# @title Step 2: Harvest 100k Behavioral Cloning Transitions
# Offloads the computational simulation overhead completely to the cloud instance
!python training/generate_bc_data.py \
    --num_transitions 100000 \
    --save_path data/bc_dataset/bc_data.npz \
    --max_entities 200


### ☢️ Emergency: CUDA Memory Reset
If you encounter a `CUDA Out of Memory` error, run this cell to nuke all stale tensors and environment caches before restarting Step 3 or 4.

In [ ]:
import torch
import gc
try:
    # Clear PyTorch Cache
    torch.cuda.empty_cache()
    # Force Garbage Collection
    gc.collect()
    print("✅ CUDA Memory Purged.")
    # If OOM persists, you may need to go to: 
    # Runtime -> Restart Session
except:
    pass

## 🏋️ Phase 3: Relational Supervised Pre-Training (Behavioral Cloning)
This step optimizes our model's relational projection grid layer to map source-target pairing intents. It extracts the matching ship allocation logit slices along valid paths and uses an `AdamW` optimizer regularized against zero-padding token noise to forge spatial representations.


In [ ]:
# @title Step 3: Run GPU-Accelerated Behavioral Cloning
# Batch size reduced to 16 to accommodate O(N^2) relational transformer tensors
!python training/train_bc.py \
    --npz_path data/bc_dataset/bc_data.npz \
    --epochs 10 \
    --batch_size 16 \
    --save_path checkpoints/bc_pretrained.pt


In [ ]:
# @title 🔄 Activate Background Checkpoint Sync
# Run this cell to start a background thread that syncs local checkpoints to GDrive every 5 minutes
import subprocess
import time
import threading
import os

def auto_sync():
    print("Background sync thread started.")
    while True:
        try:
            # 1. Sync PPO Checkpoints
            if os.path.exists('checkpoints'):
                subprocess.run(["cp", "-r", "checkpoints/.", "/content/drive/MyDrive/OrbitWars_Checkpoints/"], capture_output=True)
            
            # 2. Sync Expert Trajectories (BC Data)
            if os.path.exists('data/bc_dataset/bc_data.npz'):
                subprocess.run(["cp", "data/bc_dataset/bc_data.npz", "/content/drive/MyDrive/OrbitWars_Checkpoints/"], capture_output=True)
                
        except Exception as e:
            pass # Silent fail to prevent notebook clutter
        time.sleep(300) # Sync every 5 minutes

threading.Thread(target=auto_sync, daemon=True).start()
print("Background auto-sync to Google Drive activated.")

## 🚀 Phase 4: Trust-Region Reinforcement Learning Fine-Tuning
With our model's weights safely initialized with expert tactical logic, we activate the online PPO engine. The model shifts from imitating the heuristic to exploring meta-breaking strategies. Turn-by-turn action masking grids prevent illegal maneuvers, while an online Potential-Based Reward Shaper guides exploration step-by-step.


In [ ]:
# @title Step 4: Execute Relational PPO Fine-Tuning Loop
# Collects transitions online and executes trust-region clipped surrogate gradient adjustments
!python training/train_ppo.py \
    --total_timesteps 1000000 \
    --batch_size 2048 \
    --bc_checkpoint checkpoints/bc_pretrained.pt \
    --device cuda


### 🩹 Emergency Resume Protocol
If your Colab instance disconnects, your progress is safely stored in your Google Drive. 
1. Re-run **Step 1** and the **Gdrive Mount** cell.
2. Identify the latest checkpoint in your Drive (e.g., `ppo_step_400000.pt`).
3. Use the cell below to resume training from that specific point.

In [ ]:
# @title 🛠️ Step 4b: Resume Interrupted PPO Training
# Update the --checkpoint and --start_step based on your latest GDrive backup
!python training/train_ppo.py \
    --total_timesteps 1000000 \
    --batch_size 2048 \
    --checkpoint /content/drive/MyDrive/OrbitWars_Checkpoints/ppo_step_400000.pt \
    --start_step 400000 \
    --device cuda

## 🏆 Phase 5: Deterministic Slot-Invariant Tournament Evaluation
To measure true policy capacity, we evaluate our fine-tuned model deterministically (`argmax`) against fixed baseline bots. The evaluator cycles our agent through all four player seats ($0, 1, 2, 3$) to enforce complete spatial invariance and compute true ELO metrics.


In [ ]:
# @title Step 5: Evaluate Peak ELO Capacity Against Baselines
!python training/evaluation.py \
    --agent_path checkpoints/ppo_step_1000000.pt \
    --num_games 40 \
    --device cuda


### 💾 Backup Layer: Hard-Saving Checkpoints to Google Drive
Google Colab ephemeral container storage terminates automatically when idle. Run this final block to mount your personal Drive space and copy all trained weight matrices permanently.


In [ ]:
# @title Step 6: Sync Trained Models to Google Drive
from google.colab import drive
import shutil

try:
    print("Mounting Google Drive boundary...")
    drive.mount('/content/drive')
    
    source_dir = 'checkpoints/'
    dest_dir = '/content/drive/MyDrive/OrbitWars_Checkpoints/'
    
    if os.path.exists(source_dir):
        shutil.copytree(source_dir, dest_dir, dirs_exist_ok=True)
        print(f"Successfully backed up weights to: {dest_dir}")
    else:
        print("⚠️ No checkpoints found to backup yet.")
except Exception as e:
    print(f"Drive backup sequence bypassed or failed: {e}")
